# Tahap 1 — Training ulang model utama M0 (Router-P2 YOLOv8n Dual-Branch, KITTI)

Satu notebook = **satu run** (satu seed). Urutan: seed 42 dulu; seed 7 dan 2026 hanya bila seed 42 lolos kriteria lanjut (sel 9).
Jalankan lewat **Save Version → Save & Run All (Commit)** agar tetap berjalan saat browser ditutup; hasil ada di tab *Output* versi tersebut.

**Prasyarat**: GPU T4, Internet On, input dataset `kitti-2class`, dan **Tahap 0 versi terbaru lulus** (dengan fix tanda `conf` dan `gate_signal_mode='total'`).

**Konfigurasi terkunci** (diambil dari `script_training.py`, bukan diketik ulang di sini): Opsi C, `gate_signal_mode='total'`, `normalize_bg_anchors=True`,
`router_hybrid_alpha=0` (keputusan 2026-10-09; menggantikan 0.25/target 0.45 di dokumen rencana), λ akhir **0.02** (ramp epoch 51–80; diputuskan 9 Okt malam setelah run λ = 0.04 berakhir dengan ranking terbalik), `gate_value_loss_weight` 0→0.5 (epoch 11–30),
warmup router epoch 1–10, `branch_b_loss_weight=0.7`, 100 epoch, batch 16, imgsz 640, AdamW lr0 0.002, cos_lr, `save_period=10`.

**Pengaman otomatis selama training** (revisi 2026-10-09 untuk α = 0)
- **Go/no-go epoch 50** (sebelum λ naik): std logit gate > 0.05 dan batas bawah CI 95% AUC (sinyal `total`) > 0.5 → selain itu berhenti (`gate_check.json`).
- **AUC ranking tiap 10 epoch** (revisi ke-2, menggantikan aturan aktivasi 5%/95% dan aturan std/saturasi logit): AUC logit gate vs oracle di val pada model EMA dicatat ke `auc_monitor.csv`; berhenti bila batas bawah CI 95% ≤ 0.5 pada 2 pemeriksaan berturut-turut setelah epoch 30. Alasan: run M0 seed 42 dengan λ = 0.04 berakhir dengan AUC 0.417 [0.385–0.447] (ranking terbalik) di epoch 100, padahal std logit 0.106 dan saturasi 0% lolos aturan revisi ke-1.
- Loss NaN/Inf → berhenti seketika. Setiap penghentian menyimpan `weights/last_resumable_epochN.pt` dan `STOPPED.txt`.

**Melanjutkan run yang berhenti (sesi baru)**: upload arsip run (`archive_M0_seedXX.zip`) sebagai Kaggle Dataset, isi `restore_run_from` (folder dataset) dan `resume_ckpt` di sel 1. Sel 8 memulihkan folder run ke path aslinya lalu resume; log CSV disambung. Offset EMA gate dan statistik `Diff_W` mulai dari nol saat resume (dicatat di provenance).

In [ ]:
# [1] KONFIGURASI RUN — ubah seed per run; sisanya terkunci
import os, sys, json, time, shutil, subprocess, hashlib, platform, datetime
from pathlib import Path

CFG = dict(
    repo_url="https://github.com/maugi1547/thesis-Difficulty-Aware-Router.git",
    repo_dir="/kaggle/working/thesis-Difficulty-Aware-Router",   # sel 4–5 menulis ke path ini
    code_commit="5416543",        # WAJIB sama untuk ketiga seed; ganti hanya bila patch sudah di-commit (catat!)
    data_yaml="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    out_dir="/kaggle/working/tahap1",
    seed=42,                      # 42 -> lalu 7 dan 2026 bila lolos
    stop_on_nogo=True,            # go/no-go epoch 50
    activation_stop_rule=None,    # aturan lama (alpha > 0) — dimatikan, lihat revisi 2026-10-09
    health_stop_rule=None,        # revisi ke-1 (std/saturasi) — dimatikan, terbukti terlalu longgar
    auc_stop_rule=dict(every=10, after_epoch=30, patience=2),   # revisi ke-2
    resume_last=None,             # path checkpoint di sesi INI (mis. sesi yang sama terputus)
    restore_run_from=None,        # folder Kaggle Dataset berisi arsip run (sesi baru), mis. "/kaggle/input/m0-seed42-ep80"
    resume_ckpt="last_resumable_epoch80.pt",   # nama file di weights/ arsip yang di-resume
    ref_old=dict(mAP50_A=0.862, mAP50_B=0.8446, prob_std=0.164),   # run final lama (referensi kriteria lanjut)
    tol_map=0.01, act_range=(0.20, 0.70),       # act_range & prob_std kini INFORMATIF saja (alpha = 0)
    min_logit_std=0.05, max_saturated=0.20,     # INFORMATIF saja sejak revisi ke-2
)
CFG.update(globals().get("CFG_OVERRIDE", {}))          # hanya untuk uji otomatis di luar Kaggle
assert CFG["repo_dir"] == "/kaggle/working/thesis-Difficulty-Aware-Router" or "CFG_OVERRIDE" in globals()
REPO = Path(CFG["repo_dir"]); OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
CFG["RUN_SMOKE"] = False
PROV = {"started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "cfg": CFG}

def sh(cmd, cwd=None, secret=None):
    """Jalankan perintah; token disensor dari output & pesan error."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if secret:
        out = out.replace(secret, "***")
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(c if c != secret else '***' for c in cmd)}\n{out}")
    return out

print(json.dumps(CFG, indent=1))

In [ ]:
# [2] CLONE dari GitHub + checkout commit terkunci (clone baru setiap sesi -> tidak ada sisa state lama)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                                              # repo publik: token tidak perlu
url = CFG["repo_url"]
auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url

if REPO.exists():
    shutil.rmtree(REPO)
sh(["git", "clone", "--quiet", auth_url, str(REPO)], secret=token)
sh(["git", "remote", "set-url", "origin", url], cwd=REPO)          # token tidak tersimpan di .git/config
try:
    sh(["git", "checkout", "--quiet", CFG["code_commit"]], cwd=REPO)
except RuntimeError as e:
    raise RuntimeError(f"Commit {CFG['code_commit']} tidak ada di GitHub. Sudah di-push? "
                       f"HEAD remote = {sh(['git', 'log', '-1', '--format=%h %s', 'origin/HEAD'], cwd=REPO)}") from e
PROV["git_commit"] = sh(["git", "rev-parse", "HEAD"], cwd=REPO)
PROV["git_commit_msg"] = sh(["git", "log", "-1", "--format=%s (%cd)"], cwd=REPO)
PROV["git_branches_containing"] = sh(["git", "branch", "-r", "--contains", "HEAD"], cwd=REPO).split()
print("✅ commit:", PROV["git_commit"][:10], "|", PROV["git_commit_msg"], "| branch:", PROV["git_branches_containing"],
      "| token:", "ya" if token else "tidak")

In [ ]:
# [3] INSTALL editable dari repo (ultralytics pip bawaan dihapus agar tidak tertukar)
print(sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"]) or "uninstall: ok")
print(sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])[-500:] or "install: ok")

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/router_train_utils.py
# [4] ditulis oleh notebook Tahap 0 (hapus sel ini setelah file di-commit ke repo)
"""
router_train_utils.py — utilitas murni untuk training & sanity check Router-P2 Dual-Branch.

Dipakai bersama oleh script_training.py (run utama) dan notebook smoke test, supaya yang
diuji di Tahap 0 adalah kode yang sama dengan yang dipakai run 100 epoch.
Isi: scheduler berparameter, penerap ROUTER_CFG, logger CSV router (per-run, 11 kolom,
label epoch diperbaiki), logger loss + penjaga NaN, timer epoch, pemeriksa CSV/loss,
estimasi anggaran, uji kalibrasi tersimpan, uji konsistensi gate.
"""
import os, sys, csv, json, time, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch

CSV_COLS = ["Epoch", "Batch", "Lambda", "P2_Prob_Real", "Rel", "Diff_W", "Final_L3",
            "GV_Target", "GV_Loss", "GV_Weight", "GV_Offset"]
LOSS_NAMES = ["box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B"]


def find_router(model):
    """Kembalikan modul router pertama (nama kelas mengandung 'DifficultyAwareRouter')."""
    for m in model.modules():
        if "DifficultyAwareRouter" in m.__class__.__name__:
            return m
    raise RuntimeError("Router tidak ditemukan di model.")


def _ramp(epoch, start, length, final):
    """0 sampai epoch == start, lalu naik linear ke `final` selama `length` epoch, lalu tetap."""
    if epoch <= start:
        return 0.0
    return final * min((epoch - start) / max(length, 1), 1.0)


def make_curriculum_scheduler(router_warmup_end, gv_start, gv_ramp, final_gv,
                              lam_start, lam_ramp, final_lambda, history=None):
    """
    Versi berparameter dari ideal_curriculum_scheduler (script_training.py).

    Konfigurasi run utama yang IDENTIK dengan kode lama:
        make_curriculum_scheduler(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                                  lam_start=50, lam_ramp=30, final_lambda=<nilai Anda>)
    Perbedaan perilaku dgn versi lama: _is_warmup di-set dua arah (lama hanya mematikan).
    Untuk run baru hasilnya sama; untuk resume, versi ini lebih benar.

    history: dict opsional, diisi {epoch: {"lambda", "gv", "warmup"}} untuk diverifikasi.
    """
    def scheduler(trainer):
        epoch = trainer.epoch + 1
        lam = _ramp(epoch, lam_start, lam_ramp, final_lambda)
        gv = _ramp(epoch, gv_start, gv_ramp, final_gv)
        warm = epoch <= router_warmup_end
        targets = [trainer.model]
        if getattr(trainer, "ema", None):
            targets.append(trainer.ema.ema)
        for mdl in targets:
            mdl.router_penalty_lambda = lam
            mdl.gate_value_loss_weight = gv
            mdl.total_batches = len(trainer.train_loader)
            find_router(mdl)._is_warmup = warm
        if history is not None:
            history[epoch] = {"lambda": lam, "gv": gv, "warmup": warm}
        print(f"📈 [SCHEDULER] epoch {epoch} | warmup={warm} | lambda={lam:.4f} | gate_value_w={gv:.4f}")
    return scheduler


def make_router_cfg_applier(router_cfg, dump_path=None):
    """
    Callback on_pretrain_routine_end: tulis hyperparameter router secara EKSPLISIT ke model
    (bukan mengandalkan default getattr yang tersebar di loss.py & model_yolov8_router.py).
    Harus jalan sebelum batch pertama, karena DualBranchDetectionLoss membaca
    branch_b_loss_weight & gate_offset_momentum saat criterion dibuat (lazy, batch pertama).
    """
    def apply(trainer):
        for mdl in [trainer.model] + ([trainer.ema.ema] if getattr(trainer, "ema", None) else []):
            for k, v in router_cfg.items():
                setattr(mdl, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(router_cfg, indent=1))
        print("🔧 [ROUTER CFG]", router_cfg)
    return apply


def make_router_csv_logger(csv_path, allow_existing=False):
    """
    Pengganti on_train_epoch_end di script_training.py:
      - path per-run (bukan /kaggle/working/... global yang di-append lintas run),
      - menolak file yang sudah ada (cegah data run lama tercampur),
      - memverifikasi tiap baris berisi 11 kolom,
      - kolom Epoch diganti epoch ASLI (label di loss.py = debug_counter//total_batches + 4,
        salah setiap kali warmup tidak berakhir di epoch 4). Tetap 11 kolom.
    """
    csv_path = Path(csv_path)
    if csv_path.exists() and not allow_existing:      # allow_existing=True hanya untuk resume run yang sama
        raise FileExistsError(f"{csv_path} sudah ada — hapus/ganti nama dulu.")

    def on_train_epoch_end(trainer):
        buf = getattr(trainer.model, "router_buffer", None)
        if not buf:
            return
        epoch = trainer.epoch + 1
        new = not csv_path.exists()
        with open(csv_path, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(CSV_COLS)
            for row in buf:
                if len(row) != len(CSV_COLS):
                    raise ValueError(f"router_buffer berisi {len(row)} kolom, header {len(CSV_COLS)}: {row}")
                w.writerow([epoch] + list(row[1:]))
        buf.clear()
    return on_train_epoch_end


def make_loss_items_logger(store, csv_path=None):
    """
    on_train_batch_end: simpan 8 komponen loss per batch + GAGAL CEPAT bila NaN/Inf.
    (Ultralytics 8.3.240 punya NaN recovery yang diam-diam memuat ulang checkpoint.)
    Bila csv_path diberikan, `store` ditulis ke file & dikosongkan lewat .flush(trainer)
    (dipasang di on_train_epoch_end) supaya run 100 epoch tidak menumpuk di RAM.
    """
    state = {"header": csv_path is not None and not Path(csv_path).exists()}

    def on_train_batch_end(trainer):
        li = trainer.loss_items.detach().float().cpu()
        epoch = trainer.epoch + 1
        store.append([epoch] + li.tolist())
        if not torch.isfinite(li).all() or not torch.isfinite(trainer.loss.detach()).all():
            raise FloatingPointError(f"Loss non-finite di epoch {epoch}, batch #{len(store)}: "
                                     f"{dict(zip(LOSS_NAMES, li.tolist()))}")

    def flush(trainer):
        if csv_path is None or not store:
            return
        pd.DataFrame(store, columns=["epoch"] + LOSS_NAMES).to_csv(
            csv_path, mode="a", header=state["header"], index=False)
        state["header"] = False
        store.clear()

    on_train_batch_end.flush = flush
    return on_train_batch_end


class EpochTimer:
    """
    Ukur waktu per epoch, dipisah: train (loop batch), val_total (validasi + save + callback),
    dan valB (validasi Branch B, diukur lewat wrapper `timed`).
    cuda.synchronize() supaya kernel async tidak 'bocor' ke fase berikutnya.
    """
    def __init__(self):
        self.rows, self._t, self._valB = [], {}, 0.0

    @staticmethod
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return time.perf_counter()

    def on_train_epoch_start(self, trainer):
        self._t["t0"] = self.now()
        self._valB = 0.0

    def on_train_epoch_end(self, trainer):
        self._t["t1"] = self.now()

    def on_fit_epoch_end(self, trainer):
        # final_eval() Ultralytics memanggil on_fit_epoch_end SEKALI LAGI setelah loop
        # (validasi best.pt) -> abaikan panggilan yang tidak didahului on_train_epoch_end.
        if self._t.get("t1") is None:
            return
        t2 = self.now()
        self.rows.append(dict(epoch=trainer.epoch + 1,
                              n_batches=len(trainer.train_loader),
                              train_s=self._t["t1"] - self._t["t0"],
                              val_total_s=t2 - self._t["t1"],
                              valB_s=self._valB))
        self._t["t1"] = None

    def timed(self, cb):
        def wrapper(validator):
            t0 = self.now()
            cb(validator)
            self._valB += self.now() - t0
        return wrapper


# ---------------------------- pemeriksa ----------------------------
class Report:
    def __init__(self):
        self.items = []

    def add(self, name, status, detail=""):
        icon = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}[status]
        print(f"{icon} [{status}] {name}" + (f" — {detail}" if detail else ""))
        self.items.append({"check": name, "status": status, "detail": str(detail)})

    def check(self, name, ok, detail="", warn_only=False):
        self.add(name, "PASS" if ok else ("WARN" if warn_only else "FAIL"), detail)

    @property
    def n_fail(self):
        return sum(i["status"] == "FAIL" for i in self.items)


def check_router_csv(df, n_batches, epochs, router_warmup_end, sched_history,
                     offset_clip=1.0, report=None):
    """
    Semua pemeriksaan pada router_loss_components.csv. `df` dibaca dgn pd.read_csv.

    Artefak yang DIKETAHUI dan diperhitungkan:
      - Sejak AUDIT-FIX #6 kolom GV_* berasal dari step yang SAMA (tidak lagi tertinggal 1 step),
        jadi pemeriksaan di sini ketat untuk semua baris. Dengan kode lama, baris pertama
        pasca-warmup akan gagal di cek GV_Weight (nilai warmup = 0).
      - GV_Offset adalah offset yang DIPAKAI step ini (nilai EMA sebelum di-update). Baris
        pertama pasca-warmup = 0 (EMA belum terinisialisasi), ini normal.
      - GV_Offset dicetak 4 desimal -> perubahan EMA < 5e-5 tampak 'tidak berubah'.
      - Mode 'small': batch tanpa gambar valid (tidak ada objek kecil di KEDUA branch)
        -> GV_Loss = GV_Target = 0 dan offset tidak di-update.
    """
    r = report or Report()
    r.check("CSV: header 11 kolom dgn urutan benar", list(df.columns) == CSV_COLS, list(df.columns))
    if list(df.columns) != CSV_COLS:
        return r

    exp_rows = (epochs - router_warmup_end) * n_batches
    r.check("CSV: jumlah baris = epoch pasca-warmup × batch/epoch", len(df) == exp_rows,
            f"{len(df)} vs {exp_rows}")
    r.check("CSV: tidak ada baris pada epoch warmup", int((df.Epoch <= router_warmup_end).sum()) == 0)

    num = df[CSV_COLS[2:]].astype(float)
    r.check("CSV: semua nilai finite (tidak NaN/Inf)", bool(np.isfinite(num.values).all()))

    # --- jadwal tercatat sesuai scheduler ---
    for e, g in df.groupby("Epoch"):
        h = sched_history.get(int(e))
        if h is None:
            r.add(f"E{e}: riwayat scheduler", "FAIL", "epoch tidak ada di riwayat scheduler")
            continue
        r.check(f"E{e}: kolom Lambda = jadwal ({h['lambda']:.4f})",
                np.allclose(g.Lambda.astype(float), h["lambda"], atol=1e-4))
        r.check(f"E{e}: kolom GV_Weight = jadwal ({h['gv']:.4f})",
                np.allclose(g.GV_Weight.astype(float), h["gv"], atol=1e-4))
        if h["lambda"] == 0:
            r.check(f"E{e}: Final_L3 = 0 saat lambda = 0", bool((g.Final_L3.astype(float) == 0).all()))
        else:
            r.check(f"E{e}: Final_L3 > 0 saat lambda > 0", bool((g.Final_L3.astype(float) > 0).all()),
                    f"min={g.Final_L3.min():.6f}")

    # --- rentang yang dijamin oleh kode ---
    r.check("P2_Prob_Real di [0, 1]", bool(df.P2_Prob_Real.between(0, 1).all()))
    r.check("Diff_W di [0.1, 2.0] (clamp di compute_router_loss)", bool(df.Diff_W.between(0.1 - 1e-6, 2.0 + 1e-6).all()),
            f"[{df.Diff_W.min():.4f}, {df.Diff_W.max():.4f}]")
    r.check("Rel ≥ 0", bool((df.Rel >= 0).all()))
    r.check("|GV_Loss| ≤ clip advantage (mean(p·adv), |adv| ≤ clip)",
            bool((df.GV_Loss.abs() <= offset_clip + 1e-6).all()), f"max |GV_Loss| = {df.GV_Loss.abs().max():.4f}")

    # --- kolom GV berubah tiap batch (baris pertama: offset EMA belum ada) ---
    g = df.iloc[1:].reset_index(drop=True)
    empty = ((g.GV_Loss == 0) & (g.GV_Target == 0)).mean()
    r.check("Batch tanpa sampel valid (GV_Loss = GV_Target = 0) < 5%", empty < 0.05, f"{empty:.1%}", warn_only=True)
    for col, thr in [("GV_Target", 0.95), ("GV_Loss", 0.95), ("GV_Offset", 0.80)]:
        frac = float((g[col].diff().iloc[1:] != 0).mean()) if len(g) > 1 else 0.0
        r.check(f"{col} berubah antar-batch (≥ {thr:.0%} batch)", frac >= thr, f"{frac:.1%}")

    # --- GV_Offset tidak menempel di ±clip ---
    at_clip = float((df.GV_Offset.abs() >= offset_clip - 1e-3).mean())
    r.check(f"GV_Offset tidak menempel di ±{offset_clip}", at_clip == 0, f"{at_clip:.1%} baris di batas")
    r.check("GV_Offset jauh dari batas (|offset| < 0.5·clip)", bool(df.GV_Offset.abs().max() < 0.5 * offset_clip),
            f"rentang [{df.GV_Offset.min():.4f}, {df.GV_Offset.max():.4f}]", warn_only=True)
    tail = df.GV_Offset.iloc[-max(len(df) // 4, 2):]
    r.check("GV_Offset tidak monoton naik/turun terus di 25% akhir (indikasi drift)",
            not (tail.diff().dropna() > 0).all() and not (tail.diff().dropna() < 0).all(),
            f"awal {tail.iloc[0]:.4f} → akhir {tail.iloc[-1]:.4f}", warn_only=True)

    # --- aktivasi tidak kolaps (indikator awal saja, 1 epoch tidak konklusif) ---
    last_e = df.Epoch.max()
    p_last = float(df[df.Epoch == last_e].P2_Prob_Real.mean())
    r.check(f"E{last_e}: rata-rata P2_Prob_Real di [0.05, 0.95] (bukan kolaps)", 0.05 <= p_last <= 0.95,
            f"{p_last:.3f}", warn_only=True)
    return r


def check_loss_items(li_df, router_warmup_end, sched_history, report=None):
    """Pemeriksaan 8 komponen loss per batch (loss_items.csv)."""
    r = report or Report()
    vals = li_df[LOSS_NAMES].astype(float)
    r.check("loss_items: 8 komponen finite di semua batch", bool(np.isfinite(vals.values).all()))
    for e, g in li_df.groupby("epoch"):
        lam = sched_history.get(int(e), {}).get("lambda", 0.0)
        warm = int(e) <= router_warmup_end
        rmean = float(g.router.mean())
        if warm or lam == 0:
            r.check(f"E{e}: komponen 'router' = 0 (warmup / lambda 0)", bool((g.router == 0).all()), f"mean={rmean:.6f}")
        else:
            r.check(f"E{e}: komponen 'router' > 0 (lambda > 0)", rmean > 0, f"mean={rmean:.6f}")
    # informatif (bukan kriteria lulus): 3 epoch terlalu pendek untuk menilai konvergensi
    print(li_df.groupby("epoch")[LOSS_NAMES].mean().round(4).to_string())
    return r


def budget_table(timing_df, plan, branch_b_interval=10, overhead=0.10, session_limit_h=12.0):
    """
    Estimasi jam GPU per run dari waktu epoch 'steady' (epoch ≥ 2; epoch 1 memuat cache,
    autotune cuDNN, check_amp, dsb).

    jam = (E·(t_train + t_valA) + floor(E/interval_B)·t_valB) · (1 + overhead) / 3600
    t_valA = val_total − valB (validasi Branch A + simpan checkpoint + callback lain).
    """
    st = timing_df[timing_df.epoch >= 2]
    if st.empty:
        st = timing_df
    t_train = st.train_s.mean()
    t_valB = st.valB_s.mean()
    t_valA = (st.val_total_s - st.valB_s).mean()
    rows = []
    for name, (E, n_runs) in plan.items():
        h = (E * (t_train + t_valA) + (E // branch_b_interval) * t_valB) * (1 + overhead) / 3600
        rows.append(dict(run=name, epochs=E, n_runs=n_runs, jam_per_run=round(h, 2),
                         jam_total=round(h * n_runs, 2),
                         perlu_resume=h > session_limit_h - 0.5))
    summary = dict(t_train_s=round(t_train, 1), t_valA_s=round(t_valA, 1), t_valB_s=round(t_valB, 1),
                   epoch1_train_s=round(float(timing_df.iloc[0].train_s), 1))
    return pd.DataFrame(rows), summary


CALIB_T_TEST = 1.2345678901234567   # nilai "tidak bulat" -> pembulatan fp16/fp32 pasti ketahuan
CALIB_B_TEST = -0.3456789012345678


def _calib(router):
    return float(getattr(router, "calib_T", 1.0)), float(getattr(router, "calib_b", 0.0))


def test_calibration_persistence(model, build_fresh_model, tmp_dir, report=None):
    """
    Uji 0.4. Set calib_T/calib_b, simpan, muat ulang lewat 3 jalur yang benar-benar dipakai:
      J1 pickle checkpoint Ultralytics : torch.save({'ema': deepcopy(model).half()}) -> torch.load
         (= trainer.save_model + attempt_load_one_weight)
      J2 state_dict -> model baru       : load_state_dict(strict=False)
         (= load_dualbranch() di notebook uji-akurasi-latensi)
      J3 Ultralytics model.load()       : intersect_dicts + load_state_dict
         (= DualBranchTrainer.get_model(weights=...), mis. fine-tune)

    Sejak AUDIT-FIX #9 kalibrasi disimpan sebagai buffer int64 dalam mikro-unit, jadi nilai
    yang tersimpan = round(v·1e6)/1e6. "Sama persis" di sini berarti: nilai yang DIBACA
    sebelum simpan == nilai setelah muat ulang (==, bukan allclose), dan galat kuantisasi
    set→baca ≤ 5e-7.
    """
    r = report or Report()
    tmp_dir = Path(tmp_dir); tmp_dir.mkdir(parents=True, exist_ok=True)
    rt = find_router(model)
    rt.calib_T, rt.calib_b = CALIB_T_TEST, CALIB_B_TEST
    ref = (rt.calib_T, rt.calib_b)
    r.check("Kalibrasi: galat kuantisasi set→baca ≤ 5e-7",
            abs(ref[0] - CALIB_T_TEST) <= 5e-7 and abs(ref[1] - CALIB_B_TEST) <= 5e-7,
            f"set=({CALIB_T_TEST}, {CALIB_B_TEST}) baca={ref}")
    keys = [k for k in model.state_dict() if "calib" in k]
    r.check("Kalibrasi ikut state_dict (buffer)", len(keys) == 2, keys)

    # J1
    p = tmp_dir / "calib_pickle.pt"
    torch.save({"ema": deepcopy(model).half()}, p)
    m1 = torch.load(p, map_location="cpu", weights_only=False)["ema"].float()
    rt1 = find_router(m1)
    r.check("Kalibrasi J1 (pickle checkpoint Ultralytics, lewat .half()) identik", _calib(rt1) == ref, f"{_calib(rt1)}")
    r.check("Buffer kalibrasi tetap int64 setelah .half()/.float()",
            all(b.dtype == torch.int64 for n, b in rt1.named_buffers() if "calib" in n))

    # J2
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
    torch.save(sd, tmp_dir / "calib_state_dict.pt")
    sd2 = torch.load(tmp_dir / "calib_state_dict.pt", map_location="cpu")
    m2 = build_fresh_model()
    missing, unexpected = m2.load_state_dict(sd2, strict=False)
    r.check("Kalibrasi J2 (state_dict -> model baru) identik", _calib(find_router(m2)) == ref,
            f"{_calib(find_router(m2))} | missing={len(missing)} unexpected={len(unexpected)}")

    # J3
    m3 = build_fresh_model()
    m3.load(m1, verbose=False)
    r.check("Kalibrasi J3 (Ultralytics model.load) identik", _calib(find_router(m3)) == ref, f"{_calib(find_router(m3))}",
            warn_only=True)   # jalur fine-tune: ikut mewarisi kalibrasi -> sadari saat fine-tune
    return r, (m1, m2, m3)


@torch.no_grad()
def gate_decisions_both_paths(model, img):
    """
    Satu forward eval penuh (model(img) -> _predict_once -> router.forward eval), tangkap input
    router via forward-pre-hook, lalu panggil compute_gate_only pada input yang SAMA.
    Returns: (dec_forward atau None, n_active_forward, dec_gate_only, prob_gate_only)
    """
    rt = find_router(model)
    cap = {}
    h = rt.register_forward_pre_hook(lambda m, inp: cap.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        model(img)
    finally:
        h.remove()
    B = img.shape[0]
    dec_fwd = getattr(rt, "_last_gate_decision", None)        # butuh patch P2
    dec_fwd = dec_fwd.float().flatten().clone() if dec_fwd is not None else None
    n_fwd = float(rt.current_activation_prob) * B
    dec_only = rt.compute_gate_only(cap["p3"], cap["p2"]).float().flatten()
    # probabilitas (untuk margin ke threshold) — rumus sama dgn compute_gate_only
    z = rt._topk_spatial_pool(rt.squeeze_p3(cap["p3"]) + rt.squeeze_p2(cap["p2"]),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(B, 2).float() / 5.0)
    prob = torch.sigmoid((l[:, 1] - l[:, 0]) * getattr(rt, "calib_T", 1.0) + getattr(rt, "calib_b", 0.0))
    return dec_fwd, n_fwd, dec_only, prob


def test_gate_consistency(model, batches, calibs, report=None):
    """
    Uji 0.5. Untuk tiap kalibrasi & tiap batch: keputusan forward(eval) == compute_gate_only.
    `calibs`: list (nama, T, b). Model harus .eval(); presisi mengikuti dtype model.
    """
    r = report or Report()
    rt = find_router(model)
    saved = _calib(rt)
    model.eval()
    try:
        for name, T, b in calibs:
            rt.calib_T, rt.calib_b = T, b
            n_img = n_mis = 0
            n_fwd_tot = n_only_tot = 0.0
            min_margin = 1.0
            weak = False
            for img in batches:
                dec_fwd, n_fwd, dec_only, prob = gate_decisions_both_paths(model, img)
                n_img += img.shape[0]
                n_fwd_tot += n_fwd
                n_only_tot += float(dec_only.sum())
                min_margin = min(min_margin, float((prob - 0.5).abs().min()))
                if dec_fwd is None:
                    weak = True
                else:
                    n_mis += int((dec_fwd.cpu() != dec_only.cpu()).sum())
            if weak:
                r.check(f"Gate [{name}] jumlah aktif forward == compute_gate_only (VERSI LEMAH, patch P2 belum)",
                        abs(n_fwd_tot - n_only_tot) < 0.5,
                        f"{n_fwd_tot:.0f} vs {n_only_tot:.0f} dari {n_img}", warn_only=True)
            else:
                r.check(f"Gate [{name}] keputusan per-gambar identik", n_mis == 0,
                        f"{n_mis}/{n_img} beda | aktif {n_only_tot:.0f}/{n_img} "
                        f"({n_only_tot / max(n_img, 1):.1%}) | margin min |p−0.5| = {min_margin:.2e}")
    finally:
        rt.calib_T, rt.calib_b = saved
    return r


def percentile_bias_for_rate(model, batches, target_rate, T=1.0):
    """b sehingga fraksi (logit·T + b > 0) ≈ target_rate pada `batches` (gaya DynamicDet).
    Dipakai HANYA untuk membuat kalibrasi uji yang menaruh banyak sampel dekat threshold."""
    rt = find_router(model)
    saved = _calib(rt)
    rt.calib_T, rt.calib_b = 1.0, 0.0
    logits = []
    for img in batches:
        _, _, _, prob = gate_decisions_both_paths(model, img)
        logits.append(torch.logit(prob.clamp(1e-7, 1 - 1e-7)).cpu())
    rt.calib_T, rt.calib_b = saved
    lg = torch.cat(logits) * T
    return float(-torch.quantile(lg, 1.0 - target_rate))




# =============================================================================
# PATCH SUMBER (EOL-aware, idempotent) — dipakai notebook sebelum import ultralytics
# =============================================================================
def apply_source_patch(path, old, new, done_marker, name):
    """
    Ganti `old` -> `new` tepat satu kali di file `path`, mempertahankan EOL file (LF/CRLF).
    Idempotent: bila `done_marker` sudah ada, file tidak diubah. Returns status string.
    """
    path = Path(path)
    raw = path.read_bytes().decode("utf-8")
    if done_marker in raw:
        return f"ℹ️ {name}: sudah ada — tidak di-patch."
    if "\r\n" in raw:
        old, new = old.replace("\n", "\r\n"), new.replace("\n", "\r\n")
    n = raw.count(old)
    if n != 1:
        raise RuntimeError(f"{name}: pola ditemukan {n}x (harus 1) di {path} — versi file berbeda, cek manual.")
    path.write_bytes(raw.replace(old, new).encode("utf-8"))
    return f"✅ {name}: di-patch."


PATCH_P2 = dict(
    name="P2 keputusan & logit gate per-gambar (block.py, eval)",
    rel="ultralytics/nn/modules/block.py",
    done_marker="self._last_gate_logit_raw = raw_logit_diff.detach()",
    old=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
    new=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n"
         "                # [TAHAP0] per-gambar: keputusan (uji konsistensi vs compute_gate_only) dan logit\n"
         "                # mentah sebelum kalibrasi (sebaran logit di log validasi)\n"
         "                self._last_gate_decision = gate_mask.view(B).detach()\n"
         "                self._last_gate_logit_raw = raw_logit_diff.detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
)

PATCH_CONF_SIGN = dict(
    name="Tanda conf di difficulty score (loss.py): - menjadi +",
    rel="ultralytics/utils/loss.py",
    done_marker="diff_score = entropy_f + var_f + conf_f",
    old=("                # TODO: verifikasi -- router.last_conf adalah UNCERTAINTY (1 - prob, tinggi =\n"
         "                # tidak yakin), sama arah dengan entropy dan var. Mengurangkannya di sini\n"
         "                # berarti ketidakpastian kelas yang lebih tinggi MENURUNKAN skor kesulitan.\n"
         "                # Apakah tanda minus ini disengaja (mis. dulu conf berarti keyakinan)?\n"
         "                diff_score = entropy_f + var_f - conf_f\n"),
    new=("                # [FIX 2026-10-09] router.last_conf adalah UNCERTAINTY (1 - prob, tinggi = tidak\n"
         "                # yakin), searah dengan entropy dan var. Tanda minus versi lama adalah salah ketik\n"
         "                # (dikonfirmasi penulis): ketiga sinyal ketidakpastian kini DIJUMLAHKAN.\n"
         "                # Run sebelum fix ini memakai entropy + var - conf (tidak sebanding langsung).\n"
         "                diff_score = entropy_f + var_f + conf_f\n"),
)


# =============================================================================
# GO/NO-GO KUALITAS RANKING GATE (mis. checkpoint epoch 50, sebelum lambda naik)
# =============================================================================
def _raw_gate_logit(rt, p3, p2):
    """Logit gate mentah (sebelum kalibrasi) — rumus identik dengan compute_gate_only."""
    z = rt._topk_spatial_pool(rt.squeeze_p3(p3) + rt.squeeze_p2(p2),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(p3.shape[0], 2).float() / 5.0)
    return l[:, 1] - l[:, 0]


def _auc_with_ci(labels, scores, n_boot=1000, seed=0):
    """AUC (Mann-Whitney, ties = 0.5) + CI 95% bootstrap persentil. labels: 1 = Branch A lebih baik."""
    from scipy.stats import rankdata
    labels = np.asarray(labels, bool); scores = np.asarray(scores, float)

    def auc(lb, sc):
        n1, n0 = lb.sum(), (~lb).sum()
        if n1 == 0 or n0 == 0:
            return np.nan
        r = rankdata(sc)
        return (r[lb].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

    a = auc(labels, scores)
    rng = np.random.default_rng(seed); n = len(labels); boots = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        boots.append(auc(labels[i], scores[i]))
    boots = np.array([b for b in boots if np.isfinite(b)])
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if len(boots) else (np.nan, np.nan))
    return float(a), float(lo), float(hi)


@torch.no_grad()
def gate_ranking_report(model, loader, device, max_batches=None, signal_mode="small",
                        min_std=0.05, n_boot=1000):
    """
    Apakah gate sudah MERANKING gambar sesuai sinyal yang melatihnya? (model beku, FP32, eval)

    Per gambar val: logit gate mentah + loss_A/loss_B per-sample (total & small, dari criterion
    yang sama dengan training). Oracle = 1 bila loss_A < loss_B. Untuk mode 'small' hanya
    gambar dengan objek kecil ter-assign di KEDUA branch yang dipakai (sama dgn valid_mask training).

    GO bila: std logit > min_std DAN batas bawah CI 95% AUC (mode sinyal training) > 0.5.
    Catatan: oracle memakai sinyal loss yang noisy — AUC menjawab "apakah gate mempelajari
    sinyal yang diberikan", bukan "apakah keputusan itu benar secara semantik".
    """
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG
    model = deepcopy(model).float().eval().to(device)
    if isinstance(getattr(model, "args", None), dict) or not hasattr(model, "args"):
        a = getattr(model, "args", {}) or {}
        model.args = get_cfg(DEFAULT_CFG, overrides={k: a[k] for k in ("box", "cls", "dfl") if k in a})
    rt = find_router(model)
    crit = model.init_criterion()
    feats, rows = {}, {k: [] for k in ("raw", "la", "lb", "las", "lbs", "nsa", "nsb")}
    h = rt.register_forward_pre_hook(lambda m, inp: feats.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            img = batch["img"].to(device).float() / 255.0
            b = {"img": img, **{k: batch[k].to(device) for k in ("cls", "bboxes", "batch_idx")}}
            det_A, det_B = model._predict_once_dual(img)
            crit.loss_A(det_A, b); crit.loss_B(det_B, b)
            rows["raw"].append(_raw_gate_logit(rt, feats["p3"], feats["p2"]).cpu())
            rows["la"].append(crit.loss_A._last_per_sample_loss.float().cpu())
            rows["lb"].append(crit.loss_B._last_per_sample_loss.float().cpu())
            rows["las"].append(crit.loss_A._last_per_sample_loss_small.float().cpu())
            rows["lbs"].append(crit.loss_B._last_per_sample_loss_small.float().cpu())
            rows["nsa"].append(crit.loss_A._last_small_obj_count.float().cpu())
            rows["nsb"].append(crit.loss_B._last_small_obj_count.float().cpu())
    finally:
        h.remove()
    R = {k: torch.cat(v).numpy() for k, v in rows.items()}
    raw = R["raw"]
    prob = 1.0 / (1.0 + np.exp(-raw))
    out = dict(n_images=int(len(raw)), logit_mean=float(raw.mean()), logit_std=float(raw.std()),
               prob_std_uncalibrated=float(prob.std()),
               logit_p10=float(np.percentile(raw, 10)), logit_p90=float(np.percentile(raw, 90)),
               frac_saturated=float((np.abs(raw) > 9.5).mean()),
               activation_uncalibrated=float((raw > 0).mean()))
    a, lo, hi = _auc_with_ci(R["la"] < R["lb"], raw, n_boot)
    out.update(total=dict(n=int(len(raw)), oracle_rate=float((R["la"] < R["lb"]).mean()), auc=a, auc_ci=[lo, hi]))
    v = (R["nsa"] > 0) & (R["nsb"] > 0)
    if v.sum() >= 20:
        a, lo, hi = _auc_with_ci(R["las"][v] < R["lbs"][v], raw[v], n_boot)
        out.update(small=dict(n=int(v.sum()), oracle_rate=float((R["las"][v] < R["lbs"][v]).mean()),
                              auc=a, auc_ci=[lo, hi]))
    key = signal_mode if signal_mode in out else "total"
    out["decision_signal"] = key
    out["go"] = bool(out["logit_std"] > min_std and out[key]["auc_ci"][0] > 0.5)
    out["reason"] = (f"std={out['logit_std']:.4f} (> {min_std}?), AUC[{key}]={out[key]['auc']:.3f} "
                     f"CI95=[{out[key]['auc_ci'][0]:.3f}, {out[key]['auc_ci'][1]:.3f}] (batas bawah > 0.5?)")
    return out


def stop_training(trainer, reason):
    """
    Hentikan training dengan rapi setelah epoch ini. final_eval() -> strip_optimizer() membuat
    last.pt TIDAK bisa di-resume (epoch=-1, optimizer dibuang), jadi salinan utuh disimpan SEKARANG
    (save_model sudah jalan untuk epoch ini). Returns path salinan.
    """
    epoch = trainer.epoch + 1
    keep = Path(trainer.wdir) / f"last_resumable_epoch{epoch}.pt"
    # final_eval() memicu on_fit_epoch_end sekali lagi SETELAH last.pt di-strip -> jangan timpa salinan
    if getattr(trainer, "stop", False) or (Path(trainer.save_dir) / "STOPPED.txt").exists():
        return str(keep)
    if Path(trainer.last).exists():
        shutil.copy(trainer.last, keep)
    (Path(trainer.save_dir) / "STOPPED.txt").write_text(f"epoch {epoch}: {reason}\nresume: {keep}\n")
    print(f"⛔ Training dihentikan di epoch {epoch}: {reason}. Untuk lanjut: resume dari {keep}")
    trainer.stop = True
    return str(keep)


def make_activation_stop_rule(gate_val_csv, after_epoch=60, low=0.05, high=0.95, patience=5):
    """
    Aturan berhenti Tahap 1: aktivasi val (EMA, tanpa kalibrasi, dari gate_val_stats.csv) < low atau
    > high selama `patience` epoch berturut-turut SETELAH `after_epoch`. Dipasang SESUDAH callback yang
    menulis gate_val_stats.csv (on_val_end -> sudah tertulis saat on_fit_epoch_end).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="first").set_index("epoch")
        win = [e for e in range(epoch - patience + 1, epoch + 1)]
        if all(e in gv.index and e > after_epoch for e in win):
            a = gv.loc[win, "act_uncalibrated"]
            if (a < low).all() or (a > high).all():
                stop_training(trainer, f"aktivasi val {a.round(3).tolist()} di luar [{low}, {high}] "
                                       f"selama {patience} epoch (epoch {win[0]}–{win[-1]})")
    return cb


def make_logit_health_stop_rule(gate_val_csv, after_epoch=10, min_std=0.05, max_sat=0.20, patience=2):
    """
    Aturan berhenti Tahap 1 versi alpha = 0 (revisi 2026-10-09, menggantikan aturan aktivasi).
    Dengan alpha = 0 aktivasi deployment dipilih post-hoc lewat persentil, jadi yang harus dijaga
    adalah PERINGKAT logit: berhenti bila std logit val < min_std ATAU fraksi logit tersaturasi
    (|z| > 9.5, batas tanh +-10) > max_sat selama `patience` epoch berturut-turut setelah after_epoch
    (setelah warmup router: selama warmup logit memang hampir konstan).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="last").set_index("epoch")
        if "logit_std" not in gv:
            return
        win = list(range(epoch - patience + 1, epoch + 1))
        if all(e in gv.index and e > after_epoch for e in win):
            w = gv.loc[win]
            bad = (w.logit_std < min_std) | (w.frac_saturated > max_sat)
            if bad.all():
                stop_training(trainer, f"kesehatan logit gagal epoch {win[0]}–{win[-1]}: std={w.logit_std.round(4).tolist()} "
                                       f"(min {min_std}), saturasi={w.frac_saturated.round(3).tolist()} (maks {max_sat})")
    return cb


def make_auc_stop_rule(trainer_ref, out_csv, every=10, after_epoch=30, patience=2, signal_mode="total",
                       max_batches=None):
    """
    Pengaman Tahap 1 revisi ke-2 (2026-10-09, setelah run M0 seed 42 λ=0.04: AUC epoch 100 = 0.417,
    terbalik). Tiap `every` epoch: gate_ranking_report pada model EMA di val -> baris ke out_csv
    (epoch, AUC, CI, std logit, aktivasi). Berhenti bila batas bawah CI 95% AUC <= 0.5 pada
    `patience` pemeriksaan berturut-turut dengan epoch > after_epoch. Pemeriksaan <= after_epoch
    tetap dicatat (informatif). Biaya: +-1 menit per pemeriksaan di T4.
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        # Catatan: JANGAN pakai trainer.stop sebagai guard — Ultralytics men-set stop=True di epoch terakhir
        # sebelum on_fit_epoch_end, sehingga pemeriksaan epoch 100 terlewat (bug di run M0_lam002_seed42).
        if epoch % every != 0 or (Path(trainer.save_dir) / "STOPPED.txt").exists():
            return
        hist = pd.read_csv(out_csv) if Path(out_csv).exists() else pd.DataFrame()
        if not hist.empty and epoch in set(hist.epoch):
            return                                   # final_eval memicu callback lagi -> jangan dobel
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        key = rep["decision_signal"]
        row = dict(epoch=epoch, auc=rep[key]["auc"], auc_lo=rep[key]["auc_ci"][0], auc_hi=rep[key]["auc_ci"][1],
                   logit_std=rep["logit_std"], logit_mean=rep["logit_mean"],
                   act_uncalibrated=rep["activation_uncalibrated"],
                   auc_small=rep.get("small", {}).get("auc", np.nan))
        hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)
        hist.to_csv(out_csv, index=False)
        print(f"\n📊 [AUC CHECK epoch {epoch}] AUC[{key}]={row['auc']:.3f} CI95=[{row['auc_lo']:.3f}, {row['auc_hi']:.3f}] "
              f"| std logit={row['logit_std']:.3f} | aktivasi={row['act_uncalibrated']:.3f}\n")
        late = hist[hist.epoch > after_epoch].sort_values("epoch").tail(patience)
        if len(late) == patience and (late.auc_lo <= 0.5).all():
            stop_training(trainer, f"ranking gate gagal: batas bawah CI AUC <= 0.5 pada epoch {late.epoch.tolist()} "
                                   f"(AUC {late.auc.round(3).tolist()})")
    return cb


def make_criterion_cfg_applier(criterion_cfg, dump_path=None):
    """
    on_pretrain_routine_end (SETELAH router cfg): buat criterion sekarang dan set atribut yang dibaca
    v8DetectionLoss dari DIRINYA SENDIRI (mis. normalize_bg_anchors), bukan dari model.
    Model.loss() hanya membuat criterion bila belum ada, jadi criterion ini yang dipakai training.
    """
    def apply(trainer):
        m = trainer.model
        m.criterion = m.init_criterion()
        for k, v in criterion_cfg.items():
            setattr(m.criterion.loss_A, k, v); setattr(m.criterion.loss_B, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(criterion_cfg, indent=1))
        print("🔧 [CRITERION CFG]", criterion_cfg)
    return apply


def make_gate_ranking_check(trainer_ref, at_epoch, out_path, signal_mode="small", stop_on_nogo=True,
                            max_batches=None):
    """
    on_fit_epoch_end: di akhir epoch `at_epoch`, jalankan gate_ranking_report pada model EMA
    (val loader trainer). NO-GO + stop_on_nogo -> trainer.stop = True (training berhenti rapi,
    last.pt tersimpan, bisa dilanjutkan dengan resume=True bila diputuskan lanjut).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if epoch != at_epoch or Path(out_path).exists():
            return
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        rep.update(epoch=epoch, stop_on_nogo=stop_on_nogo)
        Path(out_path).write_text(json.dumps(rep, indent=1))
        print(f"\n🚦 [GATE CHECK epoch {epoch}] {'GO' if rep['go'] else 'NO-GO'} — {rep['reason']}\n")
        if not rep["go"] and stop_on_nogo:
            rep["resumable_checkpoint"] = stop_training(trainer, "gate check NO-GO")
            Path(out_path).write_text(json.dumps(rep, indent=1))
    return cb

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/script_training.py
# [5] ditulis oleh notebook Tahap 0 (hapus sel ini setelah file di-commit ke repo)
"""
script_training.py — Training Router-P2 YOLOv8n Dual-Branch (Skenario 3), versi importable.

Perubahan dibanding kode_berkomentar/script_training.py (logika model/loss TIDAK berubah):
  - Section 1 lama (patch loss.py di disk lewat penggantian string) DIHAPUS: guard
    _compute_router_penalty dan default lambda sudah ada di loss.py, sehingga patch itu
    tidak pernah mengubah apa pun, dan menulis file library saat runtime berisiko.
  - Eksekusi dibungkus build_trainer() + main(); import file ini tidak memulai training.
  - ideal_curriculum_scheduler diganti make_curriculum_scheduler (router_train_utils),
    yang dengan FULL_SCHED identik dengan jadwal lama (diuji epoch 1–100) kecuali lambda
    final = 0.04 (keputusan 2026-10-09).
  - Hyperparameter router ditulis eksplisit (ROUTER_CFG); router_hybrid_alpha = 0.
  - Semua log ke save_dir run: router_loss_components.csv (11 kolom, label epoch benar),
    loss_items.csv, epoch_timing.csv, branch_b_metrics.csv, router_cfg.json, sched_history.json.
  - Penjaga NaN per batch (gagal cepat, bukan NaN recovery diam-diam Ultralytics).
  - Sebaran logit gate per validasi (gate_val_stats.csv) dan go/no-go ranking gate di akhir
    epoch lam_start (50), tepat sebelum lambda naik (gate_check.json; NO-GO -> berhenti).
  - Butuh patch sumber (notebook sel 4): P2 logit/keputusan gate per-gambar, tanda conf (+).
"""
import os
import csv
import json
from copy import copy
from pathlib import Path

import pandas as pd
import torch
from ultralytics import settings
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.model_yolov8_router import DualBranchDetectionModel

from router_train_utils import (EpochTimer, make_activation_stop_rule, make_criterion_cfg_applier,
                                make_logit_health_stop_rule, make_auc_stop_rule,
                                make_curriculum_scheduler, make_gate_ranking_check, make_loss_items_logger,
                                make_router_cfg_applier, make_router_csv_logger)

# ==============================================================================
# KONFIGURASI RUN UTAMA
# ==============================================================================
BASE_OVERRIDES = dict(
    model="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    data="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    batch=16, imgsz=640, optimizer="AdamW", lr0=0.002, cos_lr=True, weight_decay=0.0005,
    device=0, workers=4, seed=42, amp=True, verbose=True, save=True,
)
# save_period=10: epoch50.pt (go/no-go) + titik resume bila sesi Kaggle putus.
FULL_OVERRIDES = {**BASE_OVERRIDES, "epochs": 100, "project": "/kaggle/working/runs_tahap1",
                  "name": "M0_lam002_seed42", "save_period": 10}

# Fase 1 (1–50): lambda 0; router warmup 1–10; gate_value_weight 0 -> 0.5 di epoch 11–30.
# Fase 2 (51–80): lambda 0 -> 0.02 linear. Fase 3 (81–100): lambda 0.02.
# Fase 2 (51–80): lambda 0 -> final linear. Fase 3 (81–100): lambda final.
FULL_SCHED = dict(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                  lam_start=50, lam_ramp=30, final_lambda=0.02)
# λ akhir 0.02 (keputusan 2026-10-09 malam). Run pertama λ=0.04 (seed 42) berakhir dengan ranking gate
# terbalik (AUC 0.417 di epoch 100) dan kini dilaporkan sebagai ablasi.

# Keputusan 2026-10-09: alpha = 0 -> router_target_activation tidak berpengaruh; sparsity bekerja
# sebagai harga P2 (c = lambda * Diff_W / gate_value_weight); activation rate deployment dipilih
# post-hoc (operating_point_sweep.py). Nilai target tetap ditulis supaya konfigurasi eksplisit.
# gate_signal_mode='total' (rencana, konfigurasi terkunci): 'small' membuang gambar yang objek kecilnya
# hanya tertangkap P2 (bias seleksi); 'small' diuji sebagai ablasi A5.
ROUTER_CFG = dict(
    gate_signal_mode="total",
    router_hybrid_alpha=0.0,
    router_target_activation=0.35,
    router_activation_tolerance=0.10,
    router_min_activation_floor=0.05,
    branch_b_loss_weight=0.7,
    gate_offset_momentum=0.99,
)
# Pengaman revisi ke-2 (2026-10-09): AUC ranking gate di val tiap 10 epoch; berhenti bila batas bawah
# CI <= 0.5 dua pemeriksaan berturut-turut setelah epoch 30. Menggantikan aturan aktivasi (rencana awal)
# dan aturan std/saturasi logit (revisi ke-1), yang tidak menangkap ranking terbalik di run λ=0.04.
AUC_STOP_RULE = dict(every=10, after_epoch=30, patience=2)

# Atribut yang dibaca v8DetectionLoss dari dirinya sendiri (bukan dari model) -> diset ke criterion.
CRITERION_CFG = dict(normalize_bg_anchors=True)   # ablasi A6: False


# ==============================================================================
# TRAINER & VALIDATOR
# ==============================================================================
class DualBranchTrainer(DetectionTrainer):
    """DetectionTrainer yang membangun DualBranchDetectionModel dan 8 nama komponen loss."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DualBranchDetectionModel(cfg=cfg or self.args.model, ch=3, nc=self.data["nc"], verbose=verbose)
        if weights:
            model.load(weights)
        return model

    def get_validator(self):
        # Urutan = combined_items di DualBranchDetectionLoss: 5 item loss_A + 3 item loss_B.
        self.loss_names = ("box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B")
        return DetectionValidator(self.test_loader, save_dir=self.save_dir, args=copy(self.args),
                                  _callbacks=self.callbacks)


class DualBranchValidatorB(DetectionValidator):
    """Validator yang mengevaluasi Branch B: _predict_once diganti sementara, selalu dipulihkan."""

    def __call__(self, trainer=None, model=None):
        if model is not None:
            target_model = model
        elif trainer is not None:
            target_model = trainer.ema.ema if (hasattr(trainer, "ema") and trainer.ema) else trainer.model
        else:
            raise ValueError("DualBranchValidatorB butuh `trainer` atau `model`.")
        assert hasattr(target_model, "_predict_once_dual"), "Model bukan DualBranchDetectionModel."

        original = target_model._predict_once

        def branch_b_predict_once(x, profile=False, visualize=False, embed=None):
            _, det_B = target_model._predict_once_dual(x, profile, visualize, embed)
            return det_B

        target_model._predict_once = branch_b_predict_once
        try:
            return super().__call__(trainer=trainer, model=model)
        finally:
            target_model._predict_once = original


# ==============================================================================
# CALLBACK VALIDASI
# ==============================================================================
def make_router_stat_checker(trainer_ref, csv_path=None):
    """
    Aktivasi P2 + sebaran logit gate mentah di validasi, per epoch -> csv_path.
    Angka ini TANPA kalibrasi, FP16, dataloader rect -> bukan angka deployment; gunanya memantau
    apakah logit gate tersebar antar-gambar (syarat kalibrasi persentil) sepanjang training.
    Butuh patch P2 (router._last_gate_logit_raw); tanpa patch hanya aktivasi yang tercatat.
    """
    stats = {"total": 0.0, "n": 0, "logits": []}

    def on_val_start(validator):
        stats["total"], stats["n"], stats["logits"] = 0.0, 0, []

    def on_val_batch_end(validator):
        m_ = trainer_ref.ema.ema if (hasattr(trainer_ref, "ema") and trainer_ref.ema) else trainer_ref.model
        if m_ is None:
            return
        for m in m_.modules():
            if "DifficultyAwareRouter" in m.__class__.__name__:
                v = getattr(m, "current_activation_prob", None)
                if v is not None:
                    stats["total"] += v.item() if hasattr(v, "item") else float(v)
                    stats["n"] += 1
                lg = getattr(m, "_last_gate_logit_raw", None)
                if lg is not None:
                    stats["logits"].append(lg.detach().float().flatten().cpu())
                break

    def on_val_end(validator):
        if not stats["n"]:
            return
        act = stats["total"] / stats["n"]
        msg = f"P2 AKTIF: {act * 100:.2f}%"
        row = {"epoch": trainer_ref.epoch + 1, "act_uncalibrated": act}
        if stats["logits"]:
            lg = torch.cat(stats["logits"])
            row.update(n=int(lg.numel()), logit_mean=float(lg.mean()), logit_std=float(lg.std()),
                       logit_p10=float(lg.quantile(0.1)), logit_p90=float(lg.quantile(0.9)),
                       frac_saturated=float((lg.abs() > 9.5).float().mean()))
            msg += f" | logit mean={row['logit_mean']:.3f} std={row['logit_std']:.4f}"
        print(f"\n📉 [EVALUASI ROUTER] {msg}\n")
        if csv_path is not None:
            pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not Path(csv_path).exists(), index=False)

    return on_val_start, on_val_batch_end, on_val_end


def make_branch_b_checker(trainer_ref, interval=10, csv_path=None):
    """Evaluasi Branch B tiap `interval` epoch; hasil ke csv_path (default: save_dir/branch_b_metrics.csv)."""
    def on_val_end_check_branch_b(validator):
        epoch = trainer_ref.epoch + 1
        if epoch % interval != 0:
            return
        path = Path(csv_path or Path(trainer_ref.save_dir) / "branch_b_metrics.csv")
        vb = DualBranchValidatorB(dataloader=trainer_ref.test_loader, save_dir=trainer_ref.save_dir,
                                  args=copy(trainer_ref.args))
        try:
            vb(trainer=trainer_ref)
            bm = vb.metrics.box
            print(f"🔍 [BRANCH B] Epoch {epoch} | mAP50: {bm.map50:.4f} | mAP50-95: {bm.map:.4f} | "
                  f"P: {bm.mp:.4f} | R: {bm.mr:.4f}")
            new = not path.exists()
            with open(path, "a", newline="") as f:
                w = csv.writer(f)
                if new:
                    w.writerow(["epoch", "mAP50", "mAP50-95", "precision", "recall"])
                w.writerow([epoch, bm.map50, bm.map, bm.mp, bm.mr])
        except Exception as e:  # jangan hentikan training karena evaluasi tambahan
            import traceback
            print(f"⚠️  [BRANCH B CHECK GAGAL] {type(e).__name__}: {e}")
            traceback.print_exc()

    return on_val_end_check_branch_b


# ==============================================================================
# BANGUN TRAINER (dipakai run utama DAN smoke test)
# ==============================================================================
def build_trainer(overrides, sched, router_cfg=ROUTER_CFG, branch_b_interval=10,
                  gate_check_epoch="auto", stop_on_nogo=True, criterion_cfg=CRITERION_CFG,
                  activation_stop_rule=None, health_stop_rule=None, auc_stop_rule=None):
    """
    Returns (trainer, ctx). ctx berisi objek log yang bisa diperiksa setelah train():
      timer, sched_history, loss_items_cb, paths (semua di trainer.save_dir).
    Catatan: save_dir sudah ditentukan di konstruktor trainer, jadi path log aman dipakai.

    gate_check_epoch: epoch go/no-go ranking gate ("auto" = sched["lam_start"], tepat sebelum
    lambda naik; None = mati). NO-GO + stop_on_nogo -> training berhenti setelah epoch itu.
    activation_stop_rule: dict untuk make_activation_stop_rule (aturan rencana lama, untuk alpha > 0), None = mati.
    health_stop_rule: dict untuk make_logit_health_stop_rule (revisi ke-1, terbukti terlalu longgar), None = mati.
    auc_stop_rule: dict untuk make_auc_stop_rule (revisi ke-2: AUC ranking tiap N epoch), None = mati.
    """
    settings.update({"raytune": False})
    resuming = bool(overrides.get("resume"))
    trainer = DualBranchTrainer(overrides=overrides)
    sd = Path(trainer.save_dir); sd.mkdir(parents=True, exist_ok=True)
    paths = dict(router_csv=sd / "router_loss_components.csv", loss_items=sd / "loss_items.csv",
                 timing=sd / ("epoch_timing_resume.csv" if resuming else "epoch_timing.csv"), branch_b=sd / "branch_b_metrics.csv",
                 router_cfg=sd / "router_cfg.json", sched=sd / "sched_history.json",
                 gate_val=sd / "gate_val_stats.csv", gate_check=sd / "gate_check.json",
                 criterion_cfg=sd / "criterion_cfg.json", auc_monitor=sd / "auc_monitor.csv")

    sched_history, timer = {}, EpochTimer()
    li_store = []
    li_cb = make_loss_items_logger(li_store, paths["loss_items"])
    r_start, r_batch, r_end = make_router_stat_checker(trainer, paths["gate_val"])
    if gate_check_epoch == "auto":
        gate_check_epoch = sched["lam_start"]

    def save_epoch_logs(tr):
        pd.DataFrame(timer.rows).to_csv(paths["timing"], index=False)
        paths["sched"].write_text(json.dumps({str(k): v for k, v in sched_history.items()}, indent=1))

    trainer.add_callback("on_pretrain_routine_end", make_router_cfg_applier(router_cfg, paths["router_cfg"]))
    trainer.add_callback("on_pretrain_routine_end", make_criterion_cfg_applier(criterion_cfg, paths["criterion_cfg"]))
    trainer.add_callback("on_train_epoch_start", make_curriculum_scheduler(**sched, history=sched_history))
    trainer.add_callback("on_train_epoch_start", timer.on_train_epoch_start)
    trainer.add_callback("on_train_batch_end", li_cb)
    trainer.add_callback("on_train_epoch_end", timer.on_train_epoch_end)          # sebelum flush CSV
    trainer.add_callback("on_train_epoch_end", make_router_csv_logger(paths["router_csv"], allow_existing=resuming))
    trainer.add_callback("on_train_epoch_end", li_cb.flush)
    trainer.add_callback("on_val_start", r_start)
    trainer.add_callback("on_val_batch_end", r_batch)
    trainer.add_callback("on_val_end", r_end)
    trainer.add_callback("on_val_end", timer.timed(make_branch_b_checker(trainer, branch_b_interval, paths["branch_b"])))
    trainer.add_callback("on_fit_epoch_end", timer.on_fit_epoch_end)
    trainer.add_callback("on_fit_epoch_end", save_epoch_logs)
    if gate_check_epoch:
        trainer.add_callback("on_fit_epoch_end", make_gate_ranking_check(
            trainer, gate_check_epoch, paths["gate_check"], signal_mode=router_cfg.get("gate_signal_mode", "small"),
            stop_on_nogo=stop_on_nogo))
    if activation_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_activation_stop_rule(paths["gate_val"], **activation_stop_rule))
    if health_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_logit_health_stop_rule(paths["gate_val"], **health_stop_rule))
    if auc_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_auc_stop_rule(
            trainer, paths["auc_monitor"], signal_mode=router_cfg.get("gate_signal_mode", "total"), **auc_stop_rule))

    ctx = dict(timer=timer, sched_history=sched_history, loss_items_cb=li_cb, paths=paths,
               sched=sched, router_cfg=router_cfg, criterion_cfg=criterion_cfg, overrides=overrides,
               gate_check_epoch=gate_check_epoch)
    return trainer, ctx


def main():
    print("🔥 SKENARIO 3: Router-P2 YOLOv8n Dual-Branch (Difficulty-Aware Router)")
    trainer, ctx = build_trainer(FULL_OVERRIDES, FULL_SCHED, ROUTER_CFG, branch_b_interval=10,
                                 auc_stop_rule=AUC_STOP_RULE)
    trainer.train()
    return trainer, ctx


if __name__ == "__main__":
    main()

In [ ]:
# [6] PATCH SUMBER (idempotent, EOL-aware) — hapus sel ini setelah kedua patch di-commit ke repo
#   P2   block.py : simpan keputusan & logit gate mentah per-gambar di forward eval (uji 0.5 + log validasi).
#                   Di dalam blok `not torch.jit.is_tracing()` -> graf export tidak berubah.
#   CONF loss.py  : difficulty score = entropy + var + conf (tanda minus lama = salah ketik, 2026-10-09).
sys.path.insert(0, str(REPO))
import router_train_utils as U
for P_ in (U.PATCH_P2, U.PATCH_CONF_SIGN):
    print(U.apply_source_patch(REPO / P_["rel"], P_["old"], P_["new"], P_["done_marker"], P_["name"]))
PROV["git_diff_sha256"] = hashlib.sha256(sh(["git", "diff"], cwd=REPO).encode()).hexdigest()
PROV["git_status"] = sh(["git", "status", "--short"], cwd=REPO)
print("git status:\n" + PROV["git_status"])

In [ ]:
# [7] VERIFIKASI VERSI KODE — berhenti di sini bila sesi ini tidak memakai kode yang dimaksud
import inspect
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import torch, ultralytics
import ultralytics.nn.modules.block as _blk
import ultralytics.model_yolov8_router as _mr
import ultralytics.utils.loss as _ls
_src_r = inspect.getsource(_blk.UltraLightWeightDifficultyAwareRouter)
CODE_FEATURES = {
    "ultralytics di-import dari repo (bukan pip)": Path(ultralytics.__file__).resolve().is_relative_to(REPO.resolve()),
    "AUDIT-FIX #9 kalibrasi = buffer int64": "_calib_T_micro" in _src_r,
    "GroupNorm di classifier gate": "GroupNorm(1, hidden_dim // 2)" in _src_r,
    "top-k spatial pool di gate": "_topk_spatial_pool" in _src_r,
    "patch P2 keputusan + logit gate per-gambar": "_last_gate_logit_raw" in _src_r,
    "fix tanda conf (+) di difficulty score": "entropy_f + var_f + conf_f" in inspect.getsource(_ls.v8DetectionLoss),
    "AUDIT-FIX #6 kolom GV tanpa lag": hasattr(_mr.DualBranchDetectionLoss, "_flush_pending_log"),
    "AUDIT-FIX #3 advantage di-clip": "gate_adv_clip" in inspect.getsource(_mr.DualBranchDetectionLoss),
    "AUDIT-FIX #7 normalisasi bg anchor": "normalize_bg_anchors" in inspect.getsource(_ls.v8DetectionLoss),
}
for k, v in CODE_FEATURES.items():
    print(("✅ " if v else "❌ ") + k)
PROV.update(torch=torch.__version__, ultralytics=ultralytics.__version__, python=platform.python_version(),
            cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            code_features=CODE_FEATURES)
print({k: PROV[k] for k in ("torch", "ultralytics", "cuda", "gpu")})
assert all(CODE_FEATURES.values()), "Versi kode BUKAN yang dimaksud — push/clone ulang, lalu Restart & Run All."
assert torch.cuda.is_available() or "CFG_OVERRIDE" in globals(), "GPU tidak aktif (Settings → Accelerator → GPU T4)."

In [ ]:
# [8] TRAINING M0 (± 4.5 jam di T4 menurut Tahap 0)
import importlib, pandas as pd, numpy as np
import router_train_utils as U, script_training as ST
importlib.reload(U); importlib.reload(ST)

run_name = f"M0_lam{str(ST.FULL_SCHED['final_lambda']).replace('.', '')}_seed{CFG['seed']}"   # mis. M0_lam002_seed42
SCHED = CFG.get("sched_override") or ST.FULL_SCHED
if CFG["restore_run_from"]:
    # Sesi baru: pulihkan folder run dari arsip ke save_dir ASLI (tercatat di train_args checkpoint),
    # supaya Ultralytics & logger CSV melanjutkan file yang sama.
    import torch, glob
    hits = glob.glob(str(Path(CFG["restore_run_from"]) / "**" / "weights" / CFG["resume_ckpt"]), recursive=True)
    assert len(hits) == 1, f"{CFG['resume_ckpt']} ditemukan {len(hits)}x di {CFG['restore_run_from']}: {hits}"
    src_run = Path(hits[0]).parent.parent
    ck_ = torch.load(hits[0], map_location="cpu", weights_only=False)
    assert ck_.get("epoch", -1) >= 0 and ck_.get("optimizer") is not None, "Checkpoint sudah di-strip — tidak bisa di-resume."
    dest = Path(ck_["train_args"]["save_dir"])
    assert not dest.exists(), f"{dest} sudah ada di sesi ini — hapus dulu atau pakai resume_last."
    shutil.copytree(src_run, dest)
    for f in dest.glob("STOPPED.txt"):
        f.rename(dest / f"STOPPED_resolved_{datetime.datetime.utcnow():%Y%m%dT%H%M}.txt")
    PROV["restored_from"] = dict(archive=str(src_run), ckpt=hits[0], resumed_after_epoch=int(ck_["epoch"]) + 1,
                                 note="offset EMA gate & statistik Diff_W mulai dari nol saat resume")
    del ck_
    overrides = {"resume": str(dest / "weights" / CFG["resume_ckpt"])}
    print(f"♻️  Run dipulihkan ke {dest}; resume setelah epoch {PROV['restored_from']['resumed_after_epoch']}")
elif CFG["resume_last"]:
    overrides = {"resume": CFG["resume_last"]}
else:
    overrides = {**ST.FULL_OVERRIDES, "data": CFG["data_yaml"], "seed": CFG["seed"],
                 "project": str(OUT / "runs"), "name": run_name, "exist_ok": False, **CFG.get("extra_overrides", {})}
trainer, ctx = ST.build_trainer(overrides, SCHED, ST.ROUTER_CFG, branch_b_interval=10, gate_check_epoch="auto",
                                stop_on_nogo=CFG["stop_on_nogo"], activation_stop_rule=CFG["activation_stop_rule"],
                                health_stop_rule=CFG["health_stop_rule"], auc_stop_rule=CFG["auc_stop_rule"])
RUN = Path(trainer.save_dir); P = ctx["paths"]
PROV.update(run_dir=str(RUN), run_name=run_name, seed=CFG["seed"], sched=SCHED, router_cfg=ST.ROUTER_CFG,
            criterion_cfg=ST.CRITERION_CFG, overrides=overrides)
prov_path = RUN / ("provenance_resume.json" if (CFG["restore_run_from"] or CFG["resume_last"]) else "provenance.json")
json.dump(PROV, open(prov_path, "w"), indent=1, default=str)
print("Folder run:", RUN)
t0 = time.perf_counter()
trainer.train()
PROV["train_wall_h"] = round((time.perf_counter() - t0) / 3600, 2)
PROV["stopped"] = (RUN / "STOPPED.txt").read_text() if (RUN / "STOPPED.txt").exists() else None
print(f"⏱️  {PROV['train_wall_h']} jam | berhenti dini: {PROV['stopped'] or 'tidak'}")

In [ ]:
# [9] KRITERIA LANJUT (rencana Tahap 1) — dihitung di VAL; test tidak disentuh
import torch, matplotlib.pyplot as plt
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
REPORT = U.Report()
res = pd.read_csv(RUN / "results.csv"); res.columns = [c.strip() for c in res.columns]
last_ep = int(res.epoch.iloc[-1]); target_ep = int(trainer.args.epochs)
bb = pd.read_csv(P["branch_b"]).drop_duplicates("epoch", keep="first") if P["branch_b"].exists() else pd.DataFrame()
gv = pd.read_csv(P["gate_val"]).drop_duplicates("epoch", keep="first") if P["gate_val"].exists() else pd.DataFrame()
gc = json.loads(P["gate_check"].read_text()) if P["gate_check"].exists() else {}
val_loader = getattr(trainer, "test_loader", None) or trainer.get_dataloader(trainer.data["val"], batch_size=16, rank=-1, mode="val")
sig = ST.ROUTER_CFG["gate_signal_mode"]

def load_ckpt(p):
    ck = torch.load(p, map_location="cpu", weights_only=False)
    return (ck.get("ema") or ck["model"]).float().eval()

rep = {}
for tag, f in [("e50", RUN / "weights" / "epoch50.pt"), ("last", RUN / "weights" / "last.pt")]:
    if f.exists():
        rep[tag] = U.gate_ranking_report(load_ckpt(f), val_loader, DEV, signal_mode=sig)
        print(tag, rep[tag]["reason"], "| prob_std", round(rep[tag]["prob_std_uncalibrated"], 4),
              "| aktivasi", round(rep[tag]["activation_uncalibrated"], 3))
json.dump(rep, open(RUN / "gate_report_val.json", "w"), indent=1)

ref, tol = CFG["ref_old"], CFG["tol_map"]
REPORT.check(f"Training mencapai epoch {target_ep} tanpa berhenti dini", last_ep == target_ep and not PROV.get("stopped"),
             f"epoch terakhir {last_ep}; {PROV.get('stopped') or ''}" + (f" | di-resume: {PROV['restored_from']}" if PROV.get("restored_from") else ""))
REPORT.check("Go/no-go epoch 50 = GO", gc.get("go") is True, gc.get("reason", "gate_check.json tidak ada"))
if "last" in rep:
    # Kriteria alpha = 0 (revisi 2026-10-09): peringkat logit sehat
    REPORT.check(f"[informatif] Std logit val epoch {last_ep} > {CFG['min_logit_std']}", rep["last"]["logit_std"] > CFG["min_logit_std"],
                 f"{rep['last']['logit_std']:.4f} (p10={rep['last']['logit_p10']:.2f}, p90={rep['last']['logit_p90']:.2f})", warn_only=True)
    REPORT.check(f"[informatif] Logit tersaturasi epoch {last_ep} < {CFG['max_saturated']:.0%}", rep["last"]["frac_saturated"] < CFG["max_saturated"],
                 f"{rep['last']['frac_saturated']:.1%}", warn_only=True)
if P["auc_monitor"].exists():
    am = pd.read_csv(P["auc_monitor"]); print("AUC monitor:\n" + am.round(3).to_string(index=False))
    a = rep["last"]["activation_uncalibrated"]; lo, hi = CFG["act_range"]
    REPORT.check(f"[informatif, alpha=0] Aktivasi val tanpa kalibrasi di [{lo:.0%}, {hi:.0%}]", lo <= a <= hi, f"{a:.3f}", warn_only=True)
mA = float(res["metrics/mAP50(B)"].iloc[-1])
REPORT.check(f"mAP50 Branch A ≥ {ref['mAP50_A'] - tol:.4f} (lama {ref['mAP50_A']} − {tol})", mA >= ref["mAP50_A"] - tol, f"{mA:.4f}")
if not bb.empty and last_ep in set(bb.epoch):
    mB = float(bb.set_index("epoch").loc[last_ep, "mAP50"])
    REPORT.check(f"mAP50 Branch B ≥ {ref['mAP50_B'] - tol:.4f} (lama {ref['mAP50_B']} − {tol})", mB >= ref["mAP50_B"] - tol, f"{mB:.4f}")
else:
    REPORT.add("mAP50 Branch B epoch terakhir", "FAIL", "tidak ada baris Branch B di epoch terakhir")
for tag in ("e50", "last"):
    if tag in rep:
        s_ = rep[tag]["prob_std_uncalibrated"]
        REPORT.check(f"[informatif, alpha=0] Std probabilitas gate {tag} > {ref['prob_std']} (lama)", s_ > ref["prob_std"],
                     f"{s_:.4f}", warn_only=True)
        r_ = rep[tag][rep[tag]["decision_signal"]]
        REPORT.check(f"Ranking gate {tag}: batas bawah CI 95% AUC > 0.5", r_["auc_ci"][0] > 0.5,
                     f"AUC={r_['auc']:.3f} CI=[{r_['auc_ci'][0]:.3f}, {r_['auc_ci'][1]:.3f}] n={r_['n']}")

# jejak training: aktivasi & offset per epoch, sebaran logit val, mAP A vs B
df = pd.read_csv(P["router_csv"]); ep = df.groupby("Epoch").mean(numeric_only=True)
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].plot(ep.index, ep.P2_Prob_Real, label="P2_Prob_Real (train)"); ax[0, 0].plot(ep.index, ep.GV_Target, label="GV_Target")
if not gv.empty: ax[0, 0].plot(gv.epoch, gv.act_uncalibrated, label="aktivasi val (EMA)")
ax2 = ax[0, 0].twinx(); ax2.plot(ep.index, ep.Lambda, "k--", lw=0.8, label="λ"); ax[0, 0].legend(fontsize=8); ax[0, 0].set_title("Aktivasi P2")
ax[0, 1].plot(ep.index, ep.GV_Offset); ax[0, 1].set_title("GV_Offset (rata-rata per epoch)")
if not gv.empty and "logit_std" in gv: ax[1, 0].plot(gv.epoch, gv.logit_std); ax[1, 0].axhline(0.05, color="r", ls=":"); ax[1, 0].set_title("Std logit gate di val")
ax[1, 1].plot(res.epoch, res["metrics/mAP50(B)"], label="Branch A")
if not bb.empty: ax[1, 1].plot(bb.epoch, bb.mAP50, "o-", label="Branch B")
ax[1, 1].legend(); ax[1, 1].set_title("mAP50 val")
for a_ in ax.flat: a_.grid(alpha=0.3); a_.set_xlabel("epoch")
fig.tight_layout(); fig.savefig(RUN / "tahap1_trace.png", dpi=120); plt.show()

In [ ]:
# [10] RINGKASAN + ARSIP (weights: hanya epoch50/last/best, sesuai checklist reproduksibilitas)
n = {s: sum(i["status"] == s for i in REPORT.items) for s in ("PASS", "WARN", "FAIL")}
PROV.update(finished_utc=datetime.datetime.utcnow().isoformat(timespec="seconds"), criteria_counts=n)
json.dump(REPORT.items, open(RUN / "tahap1_criteria.json", "w"), indent=1)
json.dump(PROV, open(prov_path, "w"), indent=1, default=str)   # provenance.json run awal tidak ditimpa saat resume
arc = OUT / f"archive_{RUN.name}"
if arc.exists(): shutil.rmtree(arc)
shutil.copytree(RUN, arc, ignore=lambda d, fs: [f for f in fs if f.endswith(".pt") and f not in
                ("epoch50.pt", "last.pt", "best.pt") and not f.startswith("last_resumable")])
shutil.make_archive(str(arc), "zip", arc); shutil.rmtree(arc)
print(f"\nKRITERIA LANJUT {RUN.name}: {n} | commit {PROV.get('git_commit', '?')[:10]} | arsip: {arc}.zip")
for it in REPORT.items:
    if it["status"] != "PASS":
        print(f"  {it['status']}: {it['check']} — {it['detail']}")
print("➡️  LOLOS: jalankan seed 7 dan 2026 (konfigurasi identik)" if n["FAIL"] == 0
      else "⛔ TIDAK LOLOS: diagnosis dulu, jangan jalankan seed lain.")